**this is for running all 4 models (BART, BERT, MPNet, Jina Embeddings V2)**
- doing this before finishing the actual module, so we have data for the extended abstract
- testing on all 100 data inputs, with both the original and 3-word codes
- sections: 0) data setup, 1) BART, 2) BERT, 3) MPNet, 4) Jina Embeddings V2, 5) getting eval results text file
- with subsections 1) getting all the scores in text files, 2) evaluation

**0) Data Setup**

In [1]:
raw_data = open("Pain points data-ground truth.txt", "r") 
data = raw_data.read() 

data_list = data.split("\n")
for i in range(len(data_list)):
    d = data_list[i].split("\t")
    data_list[i] = d

#remove extra characters
for string in data_list:
    if "\n" in string:
        new_string = string.replace("\n", "")
        #print(string)
        i = data_list.index(string)
        data_list[i] = new_string
    if '"' in string:
        new_string = string.replace('"', "")
        i = data_list.index(string)
        data_list[i] = new_string

print(data_list) 
raw_data.close() 

testable_data = []
ground_truths = []
for p in range(len(data_list)):
    testable_data.append(data_list[p][0])
    ground_truths.append(data_list[p][1])

print(len(testable_data)) #for reference
print(testable_data)
print(len(ground_truths))
print(ground_truths)

#three_words = ["Lathe chuck overtightend", "18 V Drills too large to use with one hand", "Disposable gloves are a large size", "Miter saw is on tall table (awkward to use)", "People forget to put away the clamps", "Someone left sawdust and wood chips everywhere", "Someone squeezed through aisle and bumped user (bumped e-stop)", "Wood scraps too small to be useful", "Digging through the unlabeled cabinets looking for drill", "Trash bag is ripped", "Had to wait for epoxy to cure"]
#print(three_words) #for reference
three_words = ["Lathe chuck overtightened", "Drill too large", "Gloves size large", "Tall miter saw", "Clamps scattered around", "Sawdust, wood everywhere", "Bumped into user", "Useless wood scraps", "Unlabeled drill cabinet", "Ripped trash bag", "Epoxy wait time"]
print(three_words) #for reference
descriptions = ["The lathe chuck was tightened too much, so it was loosened with pliers", "Only 18 V drills were displayed, which are too big to use with one hand", "Only size large disposable gloves were available, so using epoxy was a struggle", "The miter saw is a bit uncomfortable to use since it’s on a tall table", "People forgot to put the clamps away, so they needed to be searched for", "The lathe was covered in sawdust and wood chips, so it was cleaned up", "the emergency stop button of the lathe was bumped, as someone squeezed by", "wood scraps were too small to be useful to anyone, so they were trashed", "Needing to dig through the unlabeled cabinets to find a smaller 12 V hand drill", "the trash bag was ripped, so thetrash was taken out and the bag was replaced", "Epoxy needs to cure for 24 hours, so going home for the day was necessary"]
print(descriptions)
full_codes = []
for i in range(len(three_words)):
    full_codes.append(three_words[i] + ": " + descriptions[i])
print(full_codes)

[['cutting wood', '0'], ['didn’t know how to use lathe', '1'], ['Finding drill', '9'], ['Taking out trash', '10'], ['Finding clamp', '5'], ['Loosening the lathe', '1'], ['Cleaning up after others', '6'], ['Equipment too big', '2'], ['Too small of wood scraps.', '8'], ['Had to take trash out.', '10'], ['Had to find a clamp.', '5'], ['Had to dig in unlabeled container for a 12v drill.', '9'], ['Got bumped while using sander, accidentally pressed emergency stop button.', '7'], ['Had to clean off messy workspace.', '6'], ['Gloves were too big.', '3'], ['People putting garbage in the scrp bin', '8'], ['People not throwing trash properly ripping the bag', '10'], ['People not putting things where they go', '5'], ['People not putting things back the way they were found', '5'], ['People leaving a mess and not cleaning up', '6'], ['bumped by person using belt sander', '7'], ['garbage bag had hole', '10'], ['sawdust left on machine', '6'], ['lathe chuck tightened too tight', '1'], ['only found 18

In [2]:
#process into a dataset 
from datasets import Dataset
from transformers.pipelines.pt_utils import KeyDataset

data_numbers = []
for i in range(len(testable_data)):
     data_numbers.append(i)

data = {
    'TD': data_numbers,
    'text': testable_data
}
dataset = Dataset.from_dict(data)
print(dataset)

Dataset({
    features: ['TD', 'text'],
    num_rows: 103
})


**1.1) BART scoring**

In [ ]:
#model set-up
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

import sys

original_stdout = sys.stdout

with open('output-3word-BART.txt', 'w') as f:
    sys.stdout = f

    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        #sequence_to_classify = testable_data[i]
        sequence_to_classify = KeyDataset(dataset, "text")[i]
        for code in three_words:
            candidate_labels = code
            results = classifier(sequence_to_classify, candidate_labels, multi_label=False) #should I keep multi_labels=True? (whats the diff)
            print(str(results["scores"][0]) + "\t" + code) #only gimme the scores

f.close()

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
#model set-up
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

import sys

original_stdout = sys.stdout

with open('output-desc-BART.txt', 'w') as f:
    sys.stdout = f

    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        #sequence_to_classify = testable_data[i]
        sequence_to_classify = KeyDataset(dataset, "text")[i]
        for code in descriptions:
            candidate_labels = code
            results = classifier(sequence_to_classify, candidate_labels, multi_label=False) #should I keep multi_labels=True? (whats the diff)
            print(str(results["scores"][0]) + "\t" + code) #only gimme the scores

f.close()

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
#model set-up
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

import sys

original_stdout = sys.stdout

with open('output-full-BART.txt', 'w') as f:
    sys.stdout = f

    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        #sequence_to_classify = testable_data[i]
        sequence_to_classify = KeyDataset(dataset, "text")[i]
        for code in full_codes:
            candidate_labels = code
            results = classifier(sequence_to_classify, candidate_labels, multi_label=False) #should I keep multi_labels=True? (whats the diff)
            print(str(results["scores"][0]) + "\t" + code) #only gimme the scores

f.close()

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


**1.2 BART evaluation**

In [3]:
#model set-up
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")


predictions = []
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #sequence_to_classify = testable_data[i]
    sequence_to_classify = KeyDataset(dataset, "text")[i]
    for j in range(len(three_words)):
        candidate_labels = three_words[j]
        results = classifier(sequence_to_classify, candidate_labels, multi_label=False) #should I keep multi_labels=True? (whats the diff)
        if (results["scores"][0] > max_score):
            idx_of_max = j+1
            max_score = results["scores"][0]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
BART_eval_3word = f1_score(ground_truths, predictions_str, average=None) 
print(BART_eval_3word)

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


['6', '7', '6', '6', '5', '6', '6', '1', '7', '7', '7', '9', '7', '5', '3', '7', '10', '7', '7', '6', '7', '10', '6', '1', '2', '7', '5', '7', '7', '6', '6', '4', '6', '10', '6', '6', '9', '7', '7', '5', '7', '6', '6', '7', '6', '2', '4', '5', '6', '7', '2', '7', '7', '3', '6', '6', '5', '10', '5', '4', '7', '7', '6', '6', '7', '3', '6', '6', '6', '7', '6', '8', '7', '6', '7', '6', '7', '1', '7', '6', '6', '3', '7', '6', '6', '7', '10', '6', '3', '6', '7', '3', '5', '7', '6', '10', '6', '7', '6', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7', '3', '5', '

In [3]:
#model set-up
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")


predictions = []
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #sequence_to_classify = testable_data[i]
    sequence_to_classify = KeyDataset(dataset, "text")[i]
    for j in range(len(descriptions)):
        candidate_labels = descriptions[j]
        results = classifier(sequence_to_classify, candidate_labels, multi_label=False) #should I keep multi_labels=True? (whats the diff)
        if (results["scores"][0] > max_score):
            idx_of_max = j+1
            max_score = results["scores"][0]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
BART_eval_desc = f1_score(ground_truths, predictions_str, average=None) 
print(BART_eval_desc)

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


['2', '1', '5', '8', '5', '2', '6', '3', '7', '8', '5', '5', '8', '6', '2', '8', '10', '7', '8', '8', '8', '2', '8', '1', '5', '7', '5', '7', '7', '5', '6', '5', '8', '8', '5', '5', '5', '8', '2', '8', '7', '5', '5', '7', '8', '2', '2', '1', '8', '8', '5', '5', '7', '2', '8', '8', '3', '8', '8', '1', '2', '8', '1', '6', '5', '3', '8', '7', '10', '7', '8', '8', '6', '6', '8', '5', '5', '1', '2', '6', '5', '2', '8', '8', '8', '7', '8', '8', '3', '2', '7', '2', '5', '7', '8', '10', '8', '8', '6', '8', '3', '10', '8']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7', '3', '5', '1',

In [3]:
#model set-up
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")


predictions = []
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #sequence_to_classify = testable_data[i]
    sequence_to_classify = KeyDataset(dataset, "text")[i]
    for j in range(len(full_codes)):
        candidate_labels = full_codes[j]
        results = classifier(sequence_to_classify, candidate_labels, multi_label=False) #should I keep multi_labels=True? (whats the diff)
        if (results["scores"][0] > max_score):
            idx_of_max = j+1
            max_score = results["scores"][0]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
BART_eval_full = f1_score(ground_truths, predictions_str, average=None) 
print(BART_eval_full)

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


['1', '1', '10', '1', '1', '2', '1', '1', '8', '1', '1', '5', '1', '6', '1', '10', '10', '10', '1', '10', '7', '10', '1', '1', '1', '1', '1', '7', '1', '5', '1', '1', '1', '10', '1', '1', '1', '1', '1', '1', '1', '10', '5', '7', '1', '1', '2', '1', '8', '1', '5', '5', '10', '1', '10', '1', '10', '10', '10', '1', '1', '1', '1', '5', '5', '3', '10', '1', '1', '7', '1', '1', '6', '6', '1', '10', '10', '1', '1', '1', '5', '1', '1', '1', '1', '1', '10', '10', '1', '1', '1', '1', '1', '1', '10', '10', '8', '1', '8', '1', '3', '1', '8']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7'

**2.1) BERT scoring**

In [ ]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("tomaarsen/static-similarity-mrl-multilingual-v1")

with open('output-3word-BERT.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(three_words)
        similarities = model.similarity(user_embeddings, code_embeddings)
        results = similarities.tolist()
        for i in range(len(three_words)): 
            print(str(results[0][i]) + "\t" + three_words[i]) 

f.close()

In [ ]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("tomaarsen/static-similarity-mrl-multilingual-v1")

with open('output-desc-BERT.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(descriptions)
        similarities = model.similarity(user_embeddings, code_embeddings)
        results = similarities.tolist()
        for i in range(len(descriptions)): 
            print(str(results[0][i]) + "\t" + descriptions[i]) 

f.close()

In [ ]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("tomaarsen/static-similarity-mrl-multilingual-v1")

with open('output-full-BERT.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(full_codes)
        similarities = model.similarity(user_embeddings, code_embeddings)
        results = similarities.tolist()
        for i in range(len(full_codes)): 
            print(str(results[0][i]) + "\t" + full_codes[i]) 

f.close()

**2.2 BERT evaluation**

In [7]:
from sentence_transformers import SentenceTransformer
import sys

#this is a sentence transformers fine tuned version of BERT trained on various databases
#https://huggingface.co/sentence-transformers/static-similarity-mrl-multilingual-v1
model = SentenceTransformer("tomaarsen/static-similarity-mrl-multilingual-v1")
predictions = []
 
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #user_embeddings = model.encode(testable_data[i])
    user_embeddings = model.encode(KeyDataset(dataset, "text")[i])
    code_embeddings = model.encode(three_words)
    similarities = model.similarity(user_embeddings, code_embeddings)
    results = similarities.tolist()
    for j in range(len(three_words)): 
        #print(results)
        if (results[0][j] > max_score):
            idx_of_max = j+1
            max_score = results[0][j]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
BERT_eval_3word = f1_score(ground_truths, predictions_str, average=None)
print(BERT_eval_3word)

['8', '1', '2', '10', '5', '1', '10', '2', '8', '10', '5', '9', '7', '10', '3', '10', '10', '5', '10', '9', '7', '10', '6', '1', '2', '10', '5', '1', '10', '2', '8', '4', '10', '10', '11', '5', '9', '1', '5', '8', '8', '8', '3', '7', '1', '2', '4', '5', '8', '2', '2', '3', '5', '3', '5', '8', '9', '10', '8', '4', '2', '7', '5', '6', '4', '3', '6', '2', '10', '1', '2', '8', '10', '1', '1', '2', '3', '1', '2', '10', '2', '3', '10', '8', '8', '7', '10', '10', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '10', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', 

In [4]:
from sentence_transformers import SentenceTransformer
import sys

#this is a sentence transformers fine tuned version of BERT trained on various databases
#https://huggingface.co/sentence-transformers/static-similarity-mrl-multilingual-v1
model = SentenceTransformer("tomaarsen/static-similarity-mrl-multilingual-v1")
predictions = []
 
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #user_embeddings = model.encode(testable_data[i])
    user_embeddings = model.encode(KeyDataset(dataset, "text")[i])
    code_embeddings = model.encode(descriptions)
    similarities = model.similarity(user_embeddings, code_embeddings)
    results = similarities.tolist()
    for j in range(len(descriptions)): 
        #print(results)
        if (results[0][j] > max_score):
            idx_of_max = j+1
            max_score = results[0][j]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
BERT_eval_desc = f1_score(ground_truths, predictions_str, average=None)
print(BERT_eval_desc)

['8', '6', '9', '10', '5', '1', '6', '3', '8', '10', '5', '9', '7', '6', '3', '10', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '5', '9', '8', '4', '10', '10', '3', '5', '9', '6', '7', '5', '5', '9', '3', '7', '6', '2', '4', '5', '8', '1', '9', '3', '1', '3', '5', '4', '3', '10', '8', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '2', '5', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '10', '8', '8', '7', '10', '8', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '6', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7', '3',

In [4]:
from sentence_transformers import SentenceTransformer
import sys

#this is a sentence transformers fine tuned version of BERT trained on various databases
#https://huggingface.co/sentence-transformers/static-similarity-mrl-multilingual-v1
model = SentenceTransformer("tomaarsen/static-similarity-mrl-multilingual-v1")
predictions = []
 
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #user_embeddings = model.encode(testable_data[i])
    user_embeddings = model.encode(KeyDataset(dataset, "text")[i])
    code_embeddings = model.encode(full_codes)
    similarities = model.similarity(user_embeddings, code_embeddings)
    results = similarities.tolist()
    for j in range(len(full_codes)): 
        #print(results)
        if (results[0][j] > max_score):
            idx_of_max = j+1
            max_score = results[0][j]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
BERT_eval_full = f1_score(ground_truths, predictions_str, average=None)
print(BERT_eval_full)

['8', '1', '9', '10', '5', '1', '10', '2', '8', '10', '5', '9', '7', '10', '3', '10', '10', '5', '5', '5', '7', '10', '6', '1', '2', '7', '5', '1', '5', '9', '8', '4', '10', '10', '11', '5', '9', '1', '7', '9', '2', '9', '3', '7', '6', '2', '4', '5', '8', '1', '2', '3', '5', '3', '5', '8', '9', '10', '8', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '2', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '10', '8', '8', '7', '10', '10', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '6', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7', 

**3.1) MPNet Scoring**

In [ ]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")

with open('output-3word-MPNet.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(three_words)
        similarities = model.similarity(user_embeddings, code_embeddings)
        results = similarities.tolist()
        for i in range(len(three_words)): 
            print(str(results[0][i]) + "\t" + three_words[i]) 

f.close()

In [ ]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")

with open('output-desc-MPNet.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(descriptions)
        similarities = model.similarity(user_embeddings, code_embeddings)
        results = similarities.tolist()
        for i in range(len(descriptions)): 
            print(str(results[0][i]) + "\t" + descriptions[i]) 

f.close()

In [ ]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")

with open('output-full-MPNet.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(full_codes)
        similarities = model.similarity(user_embeddings, code_embeddings)
        results = similarities.tolist()
        for i in range(len(full_codes)): 
            print(str(results[0][i]) + "\t" + full_codes[i]) 

f.close()

**3.2 MPNet evaluation**

In [10]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")
predictions = []
 
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #user_embeddings = model.encode(testable_data[i])
    user_embeddings = model.encode(KeyDataset(dataset, "text")[i])
    code_embeddings = model.encode(three_words)
    similarities = model.similarity(user_embeddings, code_embeddings)
    results = similarities.tolist()
    for j in range(len(three_words)): 
        #print(results)
        if (results[0][j] > max_score):
            idx_of_max = j+1
            max_score = results[0][j]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
MPNet_eval_3word = f1_score(ground_truths, predictions_str, average=None)
print(MPNet_eval_3word)

['6', '1', '2', '10', '5', '1', '7', '2', '8', '10', '5', '9', '2', '6', '3', '10', '10', '6', '8', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '2', '8', '4', '10', '10', '11', '5', '2', '1', '7', '5', '4', '9', '3', '7', '6', '2', '4', '5', '8', '5', '9', '3', '6', '3', '8', '8', '8', '10', '6', '4', '2', '7', '10', '6', '2', '11', '6', '1', '10', '1', '4', '8', '10', '1', '1', '2', '3', '1', '4', '6', '2', '3', '10', '10', '7', '7', '10', '10', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '6', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7',

In [5]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")
predictions = []
 
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #user_embeddings = model.encode(testable_data[i])
    user_embeddings = model.encode(KeyDataset(dataset, "text")[i])
    code_embeddings = model.encode(descriptions)
    similarities = model.similarity(user_embeddings, code_embeddings)
    results = similarities.tolist()
    for j in range(len(descriptions)): 
        #print(results)
        if (results[0][j] > max_score):
            idx_of_max = j+1
            max_score = results[0][j]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
MPNet_eval_desc = f1_score(ground_truths, predictions_str, average=None)
print(MPNet_eval_desc)

['4', '1', '9', '10', '5', '1', '7', '2', '8', '10', '5', '9', '7', '6', '3', '10', '10', '5', '8', '8', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '10', '10', '11', '5', '2', '6', '7', '6', '7', '9', '3', '7', '6', '2', '4', '5', '8', '7', '9', '3', '7', '3', '10', '7', '9', '10', '6', '4', '2', '7', '7', '6', '9', '3', '6', '5', '10', '7', '6', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '10', '8', '7', '7', '10', '8', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '8', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7', '3

In [5]:
from sentence_transformers import SentenceTransformer
import sys

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")
predictions = []
 
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    #user_embeddings = model.encode(testable_data[i])
    user_embeddings = model.encode(KeyDataset(dataset, "text")[i])
    code_embeddings = model.encode(full_codes)
    similarities = model.similarity(user_embeddings, code_embeddings)
    results = similarities.tolist()
    for j in range(len(full_codes)): 
        #print(results)
        if (results[0][j] > max_score):
            idx_of_max = j+1
            max_score = results[0][j]
    predictions.append(idx_of_max)

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
MPNet_eval_full = f1_score(ground_truths, predictions_str, average=None)
print(MPNet_eval_full)

['6', '1', '9', '10', '5', '1', '7', '2', '8', '10', '5', '9', '7', '6', '3', '10', '10', '5', '8', '8', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '2', '6', '7', '6', '7', '9', '3', '7', '6', '2', '4', '5', '8', '7', '9', '3', '7', '3', '8', '6', '6', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '5', '10', '7', '6', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '8', '8', '7', '7', '10', '8', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '8', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7', '3', 

**4.1) Jina Embeddings V2 scoring**

In [ ]:
import sys
from transformers import AutoModel
from numpy.linalg import norm

cos_sim = lambda a,b: (a @ b.T) / (norm(a)*norm(b))
model = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-en', trust_remote_code=True) # trust_remote_code is needed to use the encode method
code_embeddings = model.encode(three_words)

results = []
with open('output-3word-JinaEmbeddingsV2.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(three_words)
        for k in range(len(three_words)):
            results.append(cos_sim(user_embeddings, code_embeddings[k]))
        for i in range(len(three_words)): 
            print(str(results[i]) + "\t" + three_words[i]) 
        results = []
        

f.close()

In [ ]:
import sys
from transformers import AutoModel
from numpy.linalg import norm

cos_sim = lambda a,b: (a @ b.T) / (norm(a)*norm(b))
model = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-en', trust_remote_code=True) # trust_remote_code is needed to use the encode method
code_embeddings = model.encode(descriptions)

results = []
with open('output-desc-JinaEmbeddingsV2.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(descriptions)
        for k in range(len(descriptions)):
            results.append(cos_sim(user_embeddings, code_embeddings[k]))
        for i in range(len(descriptions)): 
            print(str(results[i]) + "\t" + descriptions[i]) 
        results = []
        

f.close()

In [ ]:
import sys
from transformers import AutoModel
from numpy.linalg import norm

cos_sim = lambda a,b: (a @ b.T) / (norm(a)*norm(b))
model = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-en', trust_remote_code=True) # trust_remote_code is needed to use the encode method
code_embeddings = model.encode(full_codes)

results = []
with open('output-full-JinaEmbeddingsV2.txt', 'w') as f:
    sys.stdout = f
    
    for i in range(len(testable_data)): 
        print("\n")
        print("DATA: " + testable_data[i] + "\n")
        user_embeddings = model.encode(testable_data[i])
        code_embeddings = model.encode(full_codes)
        for k in range(len(full_codes)):
            results.append(cos_sim(user_embeddings, code_embeddings[k]))
        for i in range(len(full_codes)): 
            print(str(results[i]) + "\t" + full_codes[i]) 
        results = []
        

f.close()

**4.2 Jina Embeddings evaluation**

In [13]:
import sys
from transformers import AutoModel
from numpy.linalg import norm

predictions = []
cos_sim = lambda a,b: (a @ b.T) / (norm(a)*norm(b))
model = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-en', trust_remote_code=True) # trust_remote_code is needed to use the encode method
code_embeddings = model.encode(three_words)

results = []
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    user_embeddings = model.encode(testable_data[i])
    for k in range(len(three_words)):
        results.append(cos_sim(user_embeddings, code_embeddings[k]))
    for j in range(len(three_words)): 
        #print(results)
        if (results[j] > max_score):
            idx_of_max = j+1
            max_score = results[j]
    predictions.append(idx_of_max)
    results = []

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
JinaEmbeddingsV2_eval_3word = f1_score(ground_truths, predictions_str, average=None)
print(JinaEmbeddingsV2_eval_3word)

['6', '1', '2', '10', '5', '1', '7', '2', '8', '10', '5', '9', '7', '5', '3', '10', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '2', '10', '4', '10', '10', '11', '5', '2', '1', '7', '9', '5', '5', '3', '7', '6', '2', '4', '5', '8', '2', '2', '3', '5', '3', '7', '7', '9', '10', '9', '4', '2', '7', '7', '6', '2', '11', '6', '1', '10', '1', '2', '8', '10', '1', '1', '2', '3', '1', '2', '6', '2', '3', '10', '10', '9', '7', '10', '10', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '7', '1', '11', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7'

In [6]:
import sys
from transformers import AutoModel
from numpy.linalg import norm

predictions = []
cos_sim = lambda a,b: (a @ b.T) / (norm(a)*norm(b))
model = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-en', trust_remote_code=True) # trust_remote_code is needed to use the encode method
code_embeddings = model.encode(descriptions)

results = []
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    user_embeddings = model.encode(testable_data[i])
    for k in range(len(descriptions)):
        results.append(cos_sim(user_embeddings, code_embeddings[k]))
    for j in range(len(descriptions)): 
        #print(results)
        if (results[j] > max_score):
            idx_of_max = j+1
            max_score = results[j]
    predictions.append(idx_of_max)
    results = []

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
JinaEmbeddingsV2_eval_desc = f1_score(ground_truths, predictions_str, average=None)
print(JinaEmbeddingsV2_eval_desc)

['4', '1', '9', '10', '5', '1', '8', '2', '8', '10', '5', '9', '7', '6', '3', '10', '10', '5', '5', '5', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '10', '4', '10', '10', '11', '5', '2', '6', '7', '5', '7', '9', '3', '7', '6', '2', '4', '5', '8', '7', '2', '3', '7', '3', '8', '7', '6', '10', '6', '4', '2', '7', '7', '6', '9', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '10', '10', '7', '7', '10', '8', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '6', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7', '

In [6]:
import sys
from transformers import AutoModel
from numpy.linalg import norm

predictions = []
cos_sim = lambda a,b: (a @ b.T) / (norm(a)*norm(b))
model = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-en', trust_remote_code=True) # trust_remote_code is needed to use the encode method
code_embeddings = model.encode(full_codes)

results = []
for i in range(len(testable_data)): 
    max_score = 0
    idx_of_max = 0
    user_embeddings = model.encode(testable_data[i])
    for k in range(len(full_codes)):
        results.append(cos_sim(user_embeddings, code_embeddings[k]))
    for j in range(len(full_codes)): 
        #print(results)
        if (results[j] > max_score):
            idx_of_max = j+1
            max_score = results[j]
    predictions.append(idx_of_max)
    results = []

predictions_str = []
for pred in predictions:
    str_pred = str(pred)
    predictions_str.append(str_pred)
print((predictions_str))
print((ground_truths))

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

print(confusion_matrix(ground_truths, predictions_str, labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]))
JinaEmbeddingsV2_eval_full = f1_score(ground_truths, predictions_str, average=None)
print(JinaEmbeddingsV2_eval_full)

['8', '1', '9', '10', '5', '1', '10', '2', '8', '10', '5', '9', '7', '6', '3', '10', '10', '5', '5', '5', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '10', '4', '10', '10', '3', '5', '2', '6', '7', '5', '7', '9', '3', '7', '6', '2', '4', '5', '8', '1', '2', '3', '5', '3', '8', '7', '6', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '2', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '10', '10', '7', '7', '10', '8', '3', '9', '7', '3', '5', '1', '6', '10', '8', '7', '5', '1', '3', '8', '10']
['0', '1', '9', '10', '5', '1', '6', '2', '8', '10', '5', '9', '7', '6', '3', '8', '10', '5', '5', '6', '7', '10', '6', '1', '2', '7', '5', '1', '7', '9', '8', '4', '8', '10', '11', '5', '0', '6', '7', '5', '7', '0', '3', '7', '6', '2', '4', '5', '8', '0', '2', '3', '7', '3', '6', '1', '0', '10', '6', '4', '2', '7', '7', '6', '2', '3', '6', '1', '10', '7', '4', '8', '10', '6', '7', '9', '3', '1', '4', '6', '9', '3', '0', '8', '0', '7', '10', '8', '3', '9', '7', '

**5) Evaluation results**

In [ ]:
with open('eval-3word.txt', 'w') as f:
    sys.stdout = f

    print("**LONG Code, followed by the F1 score for BART, BERT, MPNet, Jina Embeddings V2**")
    print("\n")

    three_words.insert(0, "No code assigned")
    for i in range(len(three_words)):
        print(three_words[i] + "\t")
        print(BART_eval_3word[i])
        print("\t")
        print(BERT_eval_3word[i])
        print("\t")
        print(MPNet_eval_3word[i])
        print("\t")
        print(JinaEmbeddingsV2_eval_3word[i])
        print("\n")

f.close()

In [ ]:
with open('eval-desc.txt', 'w') as f:
    sys.stdout = f

    print("**SHORT Code, followed by the F1 score for BART, BERT, MPNet, Jina Embeddings V2**")
    print("\n")

    descriptions.insert(0, "No code assigned")
    for i in range(len(descriptions)):
        print(descriptions[i] + "\t")
        print(BART_eval_desc[i])
        print("\t")
        print(BERT_eval_desc[i])
        print("\t")
        print(MPNet_eval_desc[i])
        print("\t")
        print(JinaEmbeddingsV2_eval_desc[i])
        print("\n")

f.close()

In [ ]:
with open('eval-full.txt', 'w') as f:
    sys.stdout = f

    print("**SHORT Code, followed by the F1 score for BART, BERT, MPNet, Jina Embeddings V2**")
    print("\n")

    full_codes.insert(0, "No code assigned")
    for i in range(len(full_codes)):
        print(full_codes[i] + "\t")
        print(BART_eval_full[i])
        print("\t")
        print(BERT_eval_full[i])
        print("\t")
        print(MPNet_eval_full[i])
        print("\t")
        print(JinaEmbeddingsV2_eval_full[i])
        print("\n")

f.close()